In [2]:
import os
import pickle
import warnings

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

DATA_PATH = r"C:\Users\Haarun\obesity_project\data\Obesity_Dataset.tsv"
MODEL_DIR = r"C:\Users\Haarun\obesity_project\models"
RESULTS_DIR = r"C:\Users\Haarun\obesity_project\results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

features = [
    "CHR_ID",
    "CHR_POS",
    "REPORTED GENE(S)",
    "MAPPED_GENE",
    "RISK ALLELE FREQUENCY",
    "OR or BETA",
    "STRONGEST SNP-RISK ALLELE",
    "SNPS",
    "CONTEXT",
    "INTERGENIC",
    "UPSTREAM_GENE_DISTANCE",
    "DOWNSTREAM_GENE_DISTANCE"
]

categorical_columns = [
    "REPORTED GENE(S)",
    "MAPPED_GENE",
    "STRONGEST SNP-RISK ALLELE",
    "SNPS",
    "CONTEXT",
    "INTERGENIC"
]

numeric_columns = [
    "CHR_ID",
    "CHR_POS",
    "RISK ALLELE FREQUENCY",
    "OR or BETA",
    "UPSTREAM_GENE_DISTANCE",
    "DOWNSTREAM_GENE_DISTANCE"
]

print("Loading dataset...")
data = pd.read_csv(DATA_PATH, sep="\t")

data["Target"] = data["P-VALUE"].apply(lambda x: 1 if x < 1e-6 else 0)

X = data[features].copy()
y = data["Target"]

X = X.fillna("Unknown")

label_encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

for col in numeric_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X = X.fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Training Optimized MLP model...")

model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate="adaptive",
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
report = classification_report(y_test, y_pred)

print("\n===== OPTIMIZED MLP RESULTS =====")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print(report)

with open(os.path.join(MODEL_DIR, "optimized_mlp.pkl"), "wb") as f:
    pickle.dump(model, f)

with open(os.path.join(MODEL_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(MODEL_DIR, "label_encoders.pkl"), "wb") as f:
    pickle.dump(label_encoders, f)

with open(os.path.join(MODEL_DIR, "features.pkl"), "wb") as f:
    pickle.dump(features, f)

results = pd.DataFrame({
    "Model": ["Optimized MLP"],
    "Accuracy": [accuracy],
    "ROC_AUC": [roc_auc]
})

results.to_csv(
    os.path.join(RESULTS_DIR, "optimized_mlp_results.csv"),
    index=False
)

print("\nModel building completed successfully.")
print("Saved files inside models folder.")

Loading dataset...
Training Optimized MLP model...

===== OPTIMIZED MLP RESULTS =====
Accuracy: 0.8673139158576052
ROC-AUC: 0.8663727576771054
              precision    recall  f1-score   support

           0       0.25      0.39      0.31        23
           1       0.95      0.91      0.93       286

    accuracy                           0.87       309
   macro avg       0.60      0.65      0.62       309
weighted avg       0.90      0.87      0.88       309


Model building completed successfully.
Saved files inside models folder.
